In [12]:
#part-1
import numpy as np

def readVectorsSeq(filename):
    data = []
    with open(filename, 'r') as f:
        for line in f:
            point = list(map(float, line.strip().split(',')))
            data.append(np.array(point))
    return data

In [13]:
def euclidean_sq(a, b):
    return np.sum((a - b) ** 2)

In [3]:
import random

def kcenter(P, k):
    centers = []
    
    # Step 1: randomly pick first center
    centers.append(random.choice(P))
    
    for _ in range(1, k):
        # find farthest point from current centers
        max_dist = -1
        next_center = None
        
        for p in P:
            dist = min(euclidean_sq(p, c) for c in centers)
            if dist > max_dist:
                max_dist = dist
                next_center = p
        
        centers.append(next_center)
    
    return centers

In [4]:
def kmeansPP(P, k):
    centers = []
    
    # Step 1: random first center
    centers.append(random.choice(P))
    
    for _ in range(1, k):
        distances = []
        
        for p in P:
            d = min(euclidean_sq(p, c) for c in centers)
            distances.append(d)
        
        distances = np.array(distances)
        probs = distances / distances.sum()
        
        next_center = P[np.random.choice(len(P), p=probs)]
        centers.append(next_center)
    
    return centers

In [5]:
def kmeansObj(P, C):
    total = 0
    
    for p in P:
        dist = min(euclidean_sq(p, c) for c in C)
        total += dist
    
    return total / len(P)

In [11]:
import time

P = readVectorsSeq("spambase.data")

k = 10
k1 = 20

# 1. k-center
start = time.time()
C1 = kcenter(P, k)
print("kcenter time:", time.time() - start)

# 2. kmeans++
C2 = kmeansPP(P, k)
print("kmeansObj:", kmeansObj(P, C2))

# 3. kcenter + kmeans++
X = kcenter(P, k1)
C3 = kmeansPP(X, k)
print("kmeansObj (coreset):", kmeansObj(P, C3))

kcenter time: 1.6510488986968994
kmeansObj: 26522.388645336996
kmeansObj (coreset): 786373.3638766137


In [14]:
#Part-2
import re

STOP_WORDS = {"a","an","the","is","are","was","of","or","and","does","will","whose","for"}

def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', ' ', text)
    words = text.split()
    
    return [w.rstrip('s') for w in words if w not in STOP_WORDS]

In [15]:
from collections import defaultdict

class InvertedIndex:
    def __init__(self):
        self.index = defaultdict(list)
    
    def add_page(self, page_name, text):
        words = preprocess(text)
        
        for pos, word in enumerate(words):
            self.index[word].append((page_name, pos))
    
    def search_word(self, word):
        return self.index.get(word, [])

In [16]:
import math

def compute_tf(word, doc_words):
    return doc_words.count(word) / len(doc_words)

def compute_idf(word, all_docs):
    N = len(all_docs)
    count = sum(1 for doc in all_docs if word in doc)
    return math.log(N / (count + 1))

def compute_tfidf(word, doc, all_docs):
    return compute_tf(word, doc) * compute_idf(word, all_docs)

In [17]:
def search_query(query, docs):
    query_words = preprocess(query)
    scores = {}
    
    for i, doc in enumerate(docs):
        score = 0
        for word in query_words:
            score += compute_tfidf(word, doc, docs)
        scores[i] = score
    
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)

In [19]:
!pip install pyspark

     ---------------------------------------- 0.0/455.4 MB ? eta -:--:--
     ---------------------------------------- 1.0/455.4 MB 8.4 MB/s eta 0:00:55
     ---------------------------------------- 1.3/455.4 MB 2.8 MB/s eta 0:02:43
     ---------------------------------------- 2.9/455.4 MB 4.5 MB/s eta 0:01:40
     ---------------------------------------- 4.5/455.4 MB 5.4 MB/s eta 0:01:24
      --------------------------------------- 6.3/455.4 MB 5.9 MB/s eta 0:01:16
      --------------------------------------- 7.9/455.4 MB 6.3 MB/s eta 0:01:11
      --------------------------------------- 9.7/455.4 MB 6.6 MB/s eta 0:01:08
      -------------------------------------- 11.3/455.4 MB 6.8 MB/s eta 0:01:06
     - ------------------------------------- 12.3/455.4 MB 6.6 MB/s eta 0:01:08
     - ------------------------------------- 13.6/455.4 MB 6.6 MB/s eta 0:01:07
     - ------------------------------------- 14.2/455.4 MB 6.4 MB/s eta 0:01:09
     - ------------------------------------- 15

In [23]:
import numpy as np

def load_graph(filename):
    edges = set()
    
    with open(filename, 'r') as f:
        for line in f:
            u, v = map(int, line.split())
            edges.add((u, v))   # remove duplicates automatically
    
    return list(edges)

def build_graph(edges):
    graph = {}
    
    for u, v in edges:
        if u not in graph:
            graph[u] = []
        graph[u].append(v)
    
    return graph

def pagerank(graph, beta=0.8, iterations=40):
    nodes = list(graph.keys())
    N = len(nodes)
    
    ranks = {node: 1.0 / N for node in nodes}
    
    for _ in range(iterations):
        new_ranks = {node: (1 - beta)/N for node in nodes}
        
        for node in graph:
            outlinks = graph[node]
            if len(outlinks) == 0:
                continue
            
            share = ranks[node] / len(outlinks)
            
            for dest in outlinks:
                new_ranks[dest] += beta * share
        
        ranks = new_ranks
    
    return ranks

# Load graph
edges = load_graph("small.txt")   # use whole.txt later
graph = build_graph(edges)

# Run PageRank
ranks = pagerank(graph)

# Results
top5 = sorted(ranks.items(), key=lambda x: -x[1])[:5]
bottom5 = sorted(ranks.items(), key=lambda x: x[1])[:5]

print("Top 5:", top5)
print("Bottom 5:", bottom5)

Top 5: [(53, 0.0357312022326716), (14, 0.03417090697259138), (40, 0.0336300871897439), (1, 0.03000597947978862), (27, 0.02972014420140539)]
Bottom 5: [(85, 0.003409694077402821), (59, 0.003669860660127284), (81, 0.003695351749360992), (37, 0.0038082042916114506), (89, 0.003922466019802268)]


In [24]:
with open("small_output.txt", "w") as f:
    f.write("Top 5:\n")
    f.write(str(top5) + "\n\n")
    f.write("Bottom 5:\n")
    f.write(str(bottom5))

In [25]:
edges = load_graph("whole.txt")

In [26]:
graph = build_graph(edges)

In [27]:
ranks = pagerank(graph)

In [28]:
top5 = sorted(ranks.items(), key=lambda x: -x[1])[:5]
bottom5 = sorted(ranks.items(), key=lambda x: x[1])[:5]

print("Top 5:", top5)
print("Bottom 5:", bottom5)

Top 5: [(263, 0.0020202911815182184), (537, 0.0019433415714531503), (965, 0.0019254478071662634), (243, 0.0018526340162417312), (285, 0.0018273721700645148)]
Bottom 5: [(558, 0.0003286018525215298), (93, 0.0003513568937516578), (62, 0.00035314810510596285), (424, 0.0003548153864930145), (408, 0.00038779848719291705)]


In [32]:
!git clone https://github.com/varshi99-git/Big-Data.git

fatal: destination path 'Big-Data' already exists and is not an empty directory.


In [35]:
!mkdir -p Big-Data/mlbd-assignment-4

The syntax of the command is incorrect.


In [36]:
!cp *.ipynb Big-Data/mlbd-assignment-4/
!cp *.txt Big-Data/mlbd-assignment-4/

'cp' is not recognized as an internal or external command,
operable program or batch file.
'cp' is not recognized as an internal or external command,
operable program or batch file.


In [37]:
%cd Big-Data

C:\Users\saiva\Downloads\MLBD_Assignment_4\Big-Data


C:\Users\saiva\anaconda3\envs\stamp_env\lib\site-packages\IPython\core\magics\osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [38]:
!git add .

In [39]:
!git commit -m "Added MLBD Assignment 4"

On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean


In [40]:
!git push origin main

Everything up-to-date


In [41]:
import os

os.makedirs("Big-Data/mlbd-assignment-4", exist_ok=True)

In [45]:
import shutil
shutil.copy("mlbd_4_assign", "Big-Data/mlbd-assignment-4/")

FileNotFoundError: [Errno 2] No such file or directory: 'mlbd_4_assign'

In [46]:
import os
os.listdir()

['.git',
 '.gitignore',
 'Big-Data',
 'big_data_assignment_3',
 'hadoop',
 'minhash',
 'spark']

In [50]:
import os

os.makedirs("Big-Data\\mlbd-assignment-4", exist_ok=True)

In [51]:
import shutil
import os

files = os.listdir()

for file in files:
    if file.endswith(".ipynb") or file.endswith(".txt"):
        print("Copying:", file)
        shutil.copy(file, "Big-Data\\mlbd-assignment-4\\")

In [52]:
import os
os.listdir("Big-Data\\mlbd-assignment-4")

[]

In [53]:
%cd Big-Data

C:\Users\saiva\Downloads\MLBD_Assignment_4\Big-Data\Big-Data


C:\Users\saiva\anaconda3\envs\stamp_env\lib\site-packages\IPython\core\magics\osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [54]:
%cd ..

C:\Users\saiva\Downloads\MLBD_Assignment_4\Big-Data


In [55]:
import os
print(os.getcwd())

C:\Users\saiva\Downloads\MLBD_Assignment_4\Big-Data


In [56]:
os.listdir()

['.git',
 '.gitignore',
 'Big-Data',
 'big_data_assignment_3',
 'hadoop',
 'minhash',
 'spark']

In [57]:
import shutil
import os

destination = "Big-Data/mlbd-assignment-4"

os.makedirs(destination, exist_ok=True)

for file in os.listdir():
    if file.endswith(".ipynb") or file.endswith(".txt"):
        print("Copying:", file)
        shutil.copy(file, destination)

In [58]:
os.listdir("Big-Data/mlbd-assignment-4")

[]

In [59]:
import os
print(os.getcwd())

C:\Users\saiva\Downloads\MLBD_Assignment_4\Big-Data


In [60]:
os.listdir()

['.git',
 '.gitignore',
 'Big-Data',
 'big_data_assignment_3',
 'hadoop',
 'minhash',
 'spark']

In [ ]:
%cd "C:/Users/saiva/Downloads/MLBD_Assignment_4"